# Tutorial 6-3: Supervised vs. Unsupervised Dimensionality Reduction
**Course:** CSEN 140: Machine Learning/Data Mining  
**Instructor:** Dr. David C. Anastasiu

--- 
### PCA vs. LDA on the Wine Dataset

## 1. Introduction and Objectives
In this tutorial, we explore the fundamental difference between **unsupervised** and **supervised** dimensionality reduction. While both methods find a linear mapping $A: \mathbb{R}^{d} \rightarrow \mathbb{R}^{k}$, they have very different objectives:

* **Principal Component Analysis (PCA):** Focuses on **Variance**. It identifies the directions in the data that capture the most 'spread.' It is unsupervised because it completely ignores class labels.
* **Linear Discriminant Analysis (LDA):** Focuses on **Separability**. It identifies the directions that best separate different classes. It is supervised because it requires class labels to calculate 'scatter' matrices.

**Goal:** We will use the Wine dataset (13 features, 3 classes) to visualize how these two approaches 'project' the same data into 2D space.

## 2. Intuition: The Difference in 'Goals'

### Principal Component Analysis (PCA)
The goal of PCA is **reconstruction**. It tries to find a lower-dimensional subspace such that if we project the data onto it, we lose as little information (variance) as possible. It is defined by the eigenvectors of the covariance matrix $C = X^{T}X$. If two classes are mixed together but have high variance along a certain axis, PCA will preserve that axis, even if it doesn't help distinguish the classes.

### Linear Discriminant Analysis (LDA)
The goal of LDA is **discrimination**. It seeks to maximize the **Fisher Criterion**:
$$J(w) = \frac{|\mu_{1}-\mu_{2}|^{2}}{s_{1}^{2}+s_{2}^{2}}$$
Basically, LDA wants the means of the classes to be as far apart as possible (large numerator) while keeping the spread within each class as small as possible (small denominator). This 'crushes' the classes into tight, distinct clusters.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

# Load the Wine dataset from UCI
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine/wine.data"
df = pd.read_csv(url, header=None)

# Column 0 is the class label (1, 2, or 3)
y = df.iloc[:, 0].values
X = df.iloc[:, 1:].values

# PCA and LDA are sensitive to scale, so we must standardize to Unit Variance
X_std = StandardScaler().fit_transform(X)

## 3. Applying PCA (Unsupervised)
We will reduce the 13-dimensional wine data to 2 dimensions using PCA. Note that we do not pass the labels `y` to the `fit` method.

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_std)

print(f"Variance explained by PC1: {pca.explained_variance_ratio_[0]:.2%}")
print(f"Variance explained by PC2: {pca.explained_variance_ratio_[1]:.2%}")

## 4. Applying LDA (Supervised)
Now we apply LDA. Notice that for LDA, we **must** provide the labels `y` because the algorithm needs to calculate the 'within-class' and 'between-class' scatter matrices.

In [ ]:
lda = LDA(n_components=2)
X_lda = lda.fit_transform(X_std, y)

print(f"Discriminatory power of LD1: {lda.explained_variance_ratio_[0]:.2%}")
print(f"Discriminatory power of LD2: {lda.explained_variance_ratio_[1]:.2%}")

## 5. Visual Comparison
We plot both results side-by-side. Observe how the clusters differ.

In [ ]:
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
markers = ['s', 'x', 'o']

fig, ax = plt.subplots(1, 2, figsize=(14, 6))

# PCA Plot
for l, c, m in zip(np.unique(y), colors, markers):
    ax[0].scatter(X_pca[y == l, 0], X_pca[y == l, 1], color=c, label=f'Class {l}', marker=m, alpha=0.7)
ax[0].set_title("PCA: Capturing Maximum Variance")
ax[0].set_xlabel("PC1")
ax[0].set_ylabel("PC2")
ax[0].legend()

# LDA Plot
for l, c, m in zip(np.unique(y), colors, markers):
    ax[1].scatter(X_lda[y == l, 0], X_lda[y == l, 1], color=c, label=f'Class {l}', marker=m, alpha=0.7)
ax[1].set_title("LDA: Capturing Class Separability")
ax[1].set_xlabel("LD1")
ax[1].set_ylabel("LD2")
ax[1].legend()

plt.show()

## Summary
1.  **Overlap:** In the PCA plot, you might notice slight overlaps between classes (especially Classes 2 and 3). This is because PCA doesn't 'know' they are different; it only knows they share similar variance directions.
2.  **Separation:** In the LDA plot, the clusters are typically much 'tighter' and farther apart. This makes LDA an excellent pre-processing step for classification models like SVMs or Logistic Regression.